# ML Classification Pipeline - NeuroGuard

Train and evaluate classifiers to predict `overall_unsafe` from engineered session features.

**Design choices:**
- Stratified 5-fold CV (preserves ~14% minority class in each fold)
- Class-weight balancing for all models
- Models: Logistic Regression, Random Forest, Gradient Boosting, SVM, MLP
- Metrics: ROC AUC, PR AUC, F1, Accuracy, MCC
- Feature importance via permutation importance
- Final model selection and threshold tuning

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    matthews_corrcoef, make_scorer, classification_report,
    roc_curve, precision_recall_curve, confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Libraries loaded.')

## 1. Load & Prepare Data

In [ ]:
# Load assembled datasets
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')
d2d = pd.read_parquet('../data/processed/d2d_features.parquet')

# Merge features with labels
df = d2d.merge(d2c[['session_id', 'overall_unsafe']], on='session_id')

# Target
target_col = 'overall_unsafe'
y = df[target_col].astype(int).values

# Numeric features
numeric_cols = [
    'response_length_words', 'response_length_chars', 'sentence_count',
    'avg_sentence_length', 'paragraph_count', 'hedging_count', 'hedging_freq',
    'certainty_count', 'certainty_freq', 'hedging_certainty_ratio',
    'nct_id_count', 'nct_id_unique', 'citation_density',
    'bullet_count', 'numbered_list_count', 'refusal_count',
    'is_monitored', 'is_baseline_pressure'
]
# Boolean features -> int
bool_cols = ['has_refusal', 'has_citations', 'has_hedging', 'has_certainty']
for c in bool_cols:
    df[c] = df[c].astype(int)

# Categorical features -> label-encoded
cat_cols = ['scenario_id', 'pressure_id', 'monitoring_id']
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    df[f'{c}_enc'] = le.fit_transform(df[c])
    le_dict[c] = le

encoded_cat_cols = [f'{c}_enc' for c in cat_cols]

# All feature columns
feature_cols = numeric_cols + bool_cols + encoded_cat_cols
X = df[feature_cols].values.astype(float)

print(f'X shape: {X.shape}')
print(f'y distribution: {np.bincount(y)} (0=safe, 1=unsafe)')
print(f'Features ({len(feature_cols)}): {feature_cols}')

## 2. Define Models & Cross-Validation

All models use `class_weight='balanced'` (or equivalent) to handle the ~6:1 imbalance.
We use stratified 5-fold CV to preserve class proportions in every fold.

In [ ]:
# Define model pipelines (all with StandardScaler)
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            class_weight='balanced', max_iter=2000, C=1.0,
            random_state=SEED, solver='lbfgs'))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(
            class_weight='balanced', n_estimators=200, max_depth=5,
            min_samples_leaf=5, random_state=SEED))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            min_samples_leaf=5, random_state=SEED, subsample=0.8))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(
            class_weight='balanced', kernel='rbf', C=1.0,
            probability=True, random_state=SEED))
    ]),
    'MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(64, 32), max_iter=1000,
            early_stopping=True, validation_fraction=0.15,
            random_state=SEED, alpha=0.01))
    ]),
}

# Stratified 5-fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Custom scorers
scorers = {
    'roc_auc': 'roc_auc',
    'avg_precision': 'average_precision',
    'f1': 'f1',
    'accuracy': 'accuracy',
    'mcc': make_scorer(matthews_corrcoef),
}

print(f'Models defined: {list(models.keys())}')
print(f'CV strategy: {cv}')

## 3. Run Cross-Validation

In [ ]:
# Run cross-validation for all models
results = {}
for name, pipeline in models.items():
    print(f'Training: {name}...')
    cv_results = cross_validate(
        pipeline, X, y, cv=cv, scoring=scorers,
        return_train_score=False, return_estimator=True
    )
    results[name] = cv_results
    print(f'  ROC AUC: {cv_results["test_roc_auc"].mean():.3f} ± {cv_results["test_roc_auc"].std():.3f}')
    print(f'  PR AUC:  {cv_results["test_avg_precision"].mean():.3f} ± {cv_results["test_avg_precision"].std():.3f}')
    print(f'  F1:      {cv_results["test_f1"].mean():.3f} ± {cv_results["test_f1"].std():.3f}')
    print(f'  MCC:     {cv_results["test_mcc"].mean():.3f} ± {cv_results["test_mcc"].std():.3f}')
    print()

print('Cross-validation complete.')

## 4. Results Summary Table

In [ ]:
# Build summary table
summary_rows = []
for name, cv_res in results.items():
    row = {'Model': name}
    for metric in ['roc_auc', 'avg_precision', 'f1', 'accuracy', 'mcc']:
        vals = cv_res[f'test_{metric}']
        row[f'{metric}_mean'] = vals.mean()
        row[f'{metric}_std'] = vals.std()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')

# Display formatted
display_cols = []
for metric in ['roc_auc', 'avg_precision', 'f1', 'accuracy', 'mcc']:
    summary_df[metric.upper()] = summary_df.apply(
        lambda r: f'{r[f"{metric}_mean"]:.3f} ± {r[f"{metric}_std"]:.3f}', axis=1)
    display_cols.append(metric.upper())

print('=== Cross-Validation Results (5-fold Stratified) ===\n')
print(summary_df[display_cols].to_string())

# Best model by PR AUC (most informative for imbalanced data)
best_model_name = summary_df['avg_precision_mean'].idxmax()
print(f'\nBest model by PR AUC: {best_model_name} '
      f'({summary_df.loc[best_model_name, "avg_precision_mean"]:.3f})')

## 5. ROC & Precision-Recall Curves

Plot curves using out-of-fold predictions from the best model.

In [ ]:
# Collect out-of-fold predictions for ALL models
oof_preds = {}
for name, cv_res in results.items():
    y_prob_oof = np.zeros(len(y))
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        estimator = cv_res['estimator'][fold_idx]
        y_prob_oof[test_idx] = estimator.predict_proba(X[test_idx])[:, 1]
    oof_preds[name] = y_prob_oof

# Plot ROC and PR curves side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for name, y_prob in oof_preds.items():
    fpr, tpr, _ = roc_curve(y, y_prob)
    auc_val = roc_auc_score(y, y_prob)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves (Out-of-Fold)')
axes[0].legend(fontsize=8)

# PR curves
for name, y_prob in oof_preds.items():
    prec, rec, _ = precision_recall_curve(y, y_prob)
    ap_val = average_precision_score(y, y_prob)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap_val:.3f})')
axes[1].axhline(y.mean(), ls='--', color='grey', alpha=0.4, label=f'Baseline ({y.mean():.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves (Out-of-Fold)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_roc_pr_curves.png', bbox_inches='tight')
plt.show()

## 6. Confusion Matrix - Best Model

Using out-of-fold predictions at the default 0.5 threshold and an optimised F1 threshold.

In [ ]:
# Best model OOF predictions
best_probs = oof_preds[best_model_name]

# Find optimal F1 threshold
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = [f1_score(y, (best_probs >= t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)
print(f'Optimal F1 threshold: {best_threshold:.2f} (F1={best_f1:.3f})')

# Confusion matrices at default and optimal thresholds
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, (thresh, title) in enumerate([(0.5, 'Default (t=0.5)'), (best_threshold, f'Optimised (t={best_threshold:.2f})')]):
    y_pred = (best_probs >= thresh).astype(int)
    cm = confusion_matrix(y, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Safe', 'Unsafe']).plot(ax=axes[i], cmap='Blues')
    axes[i].set_title(f'{best_model_name}\n{title}')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_confusion_matrices.png', bbox_inches='tight')
plt.show()

# Classification report at optimal threshold
y_pred_opt = (best_probs >= best_threshold).astype(int)
print(f'\nClassification Report ({best_model_name}, threshold={best_threshold:.2f}):')
print(classification_report(y, y_pred_opt, target_names=['Safe', 'Unsafe']))

## 7. Feature Importance - Permutation Importance

Permutation importance is model-agnostic and measures how much each feature contributes to predictions.

In [ ]:
# Train best model on full data for permutation importance
best_pipeline = models[best_model_name]
best_pipeline.fit(X, y)

# Permutation importance (using ROC AUC as scoring)
perm_imp = permutation_importance(
    best_pipeline, X, y, n_repeats=30, random_state=SEED,
    scoring='roc_auc', n_jobs=-1
)

# Build importance DataFrame
imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std,
}).sort_values('importance_mean', ascending=False)

print(f'Feature Importance ({best_model_name}):')
print(imp_df.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
imp_plot = imp_df.head(15)  # Top 15
ax.barh(range(len(imp_plot)), imp_plot['importance_mean'], 
        xerr=imp_plot['importance_std'], color='#2980b9', alpha=0.8)
ax.set_yticks(range(len(imp_plot)))
ax.set_yticklabels(imp_plot['feature'])
ax.invert_yaxis()
ax.set_xlabel('Mean Decrease in ROC AUC')
ax.set_title(f'Permutation Feature Importance ({best_model_name})')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '12_feature_importance.png', bbox_inches='tight')
plt.show()

## 8. Model Comparison Visualization

In [ ]:
# Bar chart comparison of all models across key metrics
metrics_to_plot = ['roc_auc', 'avg_precision', 'f1', 'mcc']
metric_labels = ['ROC AUC', 'PR AUC', 'F1', 'MCC']

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

for i, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    means = [results[name][f'test_{metric}'].mean() for name in models.keys()]
    stds = [results[name][f'test_{metric}'].std() for name in models.keys()]
    x = range(len(models))
    axes[i].bar(x, means, yerr=stds, color='#2980b9', alpha=0.8, capsize=3)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels([n.split(' ')[0] for n in models.keys()], rotation=30, ha='right', fontsize=8)
    axes[i].set_title(label)
    axes[i].set_ylim(0, 1)

plt.suptitle('Model Comparison (5-Fold Stratified CV)', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '13_model_comparison.png', bbox_inches='tight')
plt.show()

## 9. Save Best Model & Summary

In [ ]:
import joblib

# Save best model
model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / 'best_classifier.joblib'
joblib.dump({
    'pipeline': best_pipeline,
    'model_name': best_model_name,
    'feature_cols': feature_cols,
    'threshold': float(best_threshold),
    'label_encoders': le_dict,
    'cv_metrics': {
        'roc_auc': float(summary_df.loc[best_model_name, 'roc_auc_mean']),
        'pr_auc': float(summary_df.loc[best_model_name, 'avg_precision_mean']),
        'f1': float(summary_df.loc[best_model_name, 'f1_mean']),
        'mcc': float(summary_df.loc[best_model_name, 'mcc_mean']),
    }
}, model_path)
print(f'Best model saved to: {model_path}')

# Final summary
print(f'\n{"="*60}')
print(f'ML CLASSIFICATION SUMMARY')
print(f'{"="*60}')
print(f'Dataset:       280 sessions ({y.sum()} unsafe, {len(y)-y.sum()} safe)')
print(f'Features:      {len(feature_cols)} ({len(numeric_cols)} numeric, {len(bool_cols)} boolean, {len(encoded_cat_cols)} categorical)')
print(f'CV Strategy:   5-fold stratified (seed={SEED})')
print(f'Best Model:    {best_model_name}')
print(f'  ROC AUC:     {summary_df.loc[best_model_name, "roc_auc_mean"]:.3f} ± {summary_df.loc[best_model_name, "roc_auc_std"]:.3f}')
print(f'  PR AUC:      {summary_df.loc[best_model_name, "avg_precision_mean"]:.3f} ± {summary_df.loc[best_model_name, "avg_precision_std"]:.3f}')
print(f'  F1:          {summary_df.loc[best_model_name, "f1_mean"]:.3f} ± {summary_df.loc[best_model_name, "f1_std"]:.3f}')
print(f'  MCC:         {summary_df.loc[best_model_name, "mcc_mean"]:.3f} ± {summary_df.loc[best_model_name, "mcc_std"]:.3f}')
print(f'  Opt. Thresh: {best_threshold:.2f} (F1={best_f1:.3f})')
print(f'Top 5 Features:')
for _, row in imp_df.head(5).iterrows():
    print(f'  - {row["feature"]:30s} importance={row["importance_mean"]:.4f}')
print(f'{"="*60}')